In [2]:
import os
import cv2
import numpy as np
import mediapipe as mp
import pandas as pd
from tqdm import tqdm

In [3]:
# === Paths ===
VIDEO_DIR = "Datasets/BdSLW60_Preprocessed"
OUTPUT_DIR = "Datasets/BdSLW60_Landmarks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# Load metadata of preprocessed clips
metadata_path = os.path.join(VIDEO_DIR, "metadata.csv")
meta_df = pd.read_csv(metadata_path)



In [5]:
# Initialize MediaPipe solutions
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh

pose_detector = mp_pose.Pose(static_image_mode=False, model_complexity=1, min_detection_confidence=0.5)
hands_detector = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)
face_detector = mp_face.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5)

In [6]:
# Face subset indices (eyes, nose, mouth corners)
FACE_SUBSET_INDICES = [
    # Outer Lips
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
    # Inner Lips
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    # Mouth Contour
    191, 80, 81, 82, 13, 312, 311, 310, 415,
    # Eyebrows (Left)
    70, 63, 105, 66, 107, 55, 65, 52, 53, 46,
    # Eyebrows (Right)
    300, 293, 334, 296, 336, 285, 295, 282, 283, 276
]

In [7]:
def extract_landmarks(video_path):
    cap = cv2.VideoCapture(video_path)
    frames_data = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR -> RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        pose_result = pose_detector.process(image)
        hands_result = hands_detector.process(image)
        face_result = face_detector.process(image)

        frame_landmarks = []

        # --- Pose ---
        if pose_result.pose_landmarks:
            for lm in pose_result.pose_landmarks.landmark:
                frame_landmarks.append([lm.x, lm.y, lm.z, lm.visibility])
        else:
            frame_landmarks.extend([[0, 0, 0, 0]] * 33)

        # --- Hands (Left + Right) ---
        for hand_label in ["Left", "Right"]:
            if hands_result.multi_hand_landmarks and hands_result.multi_handedness:
                found = False
                for hand_landmarks, handedness in zip(hands_result.multi_hand_landmarks, hands_result.multi_handedness):
                    if handedness.classification[0].label.lower() == hand_label.lower():
                        for lm in hand_landmarks.landmark:
                            frame_landmarks.append([lm.x, lm.y, lm.z, 1.0])
                        found = True
                        break
                if not found:
                    frame_landmarks.extend([[0, 0, 0, 0]] * 21)
            else:
                frame_landmarks.extend([[0, 0, 0, 0]] * 21)

        # --- Face subset ---
        if face_result.multi_face_landmarks:
            face_lm = face_result.multi_face_landmarks[0]
            for i in FACE_SUBSET_INDICES:
                lm = face_lm.landmark[i]
                frame_landmarks.append([lm.x, lm.y, lm.z, 1.0])
        else:
            frame_landmarks.extend([[0, 0, 0, 0]] * len(FACE_SUBSET_INDICES))

        frames_data.append(frame_landmarks)

    cap.release()
    return np.array(frames_data)  # Shape: (T, N, D)


In [ ]:
# === Extraction Loop ===
records = []
for i, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    video_file = row["FileName"]
    word = row["Word"]
    user = row["User"]
    trial = row["Trial"]

    video_path = os.path.join(VIDEO_DIR, word, video_file)
    if not os.path.isfile(video_path):
        print(f"⚠️ Missing video: {video_path}")
        continue

    out_dir = os.path.join(OUTPUT_DIR, word)
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, f"{user}_{word}_trial{trial}.npy")
    if os.path.exists(out_path):
        continue

    landmarks = extract_landmarks(video_path)
    np.save(out_path, landmarks)

    records.append({
        "Word": word,
        "User": user,
        "Trial": trial,
        "FileName": video_file,
        "LandmarkFile": out_path,
        "Frames": landmarks.shape[0],
        "Landmarks": landmarks.shape[1]
    })


  9%|▉         | 853/9307 [33:28<3:35:04,  1.53s/it] 

In [ ]:
# Save landmark metadata
landmark_meta_path = os.path.join(OUTPUT_DIR, "landmark_metadata.csv")
pd.DataFrame(records).to_csv(landmark_meta_path, index=False)

print(f"\n✅ Done! All landmark .npy files saved to: {OUTPUT_DIR}")
print(f"Metadata saved to: {landmark_meta_path}")